# Evaluation Analysis

**Goal:** Understand how to evaluate RAG pipeline quality using custom metrics, and how to compare experiments systematically.

We'll cover:
1. Custom metrics (latency, chunk retrieval, cost) — no LLM needed
2. How RAGAS and DeepEval metrics work conceptually
3. Visualizing metrics across experiments
4. Identifying which pipeline components matter most

In [ ]:
from src.evaluate.custom_metrics import (
    compute_chunk_metrics,
    compute_cost_estimate,
    compute_latency_metrics,
    estimate_token_count,
)

## 1. Latency Analysis

Latency is the most immediate metric — users notice slow responses. Let's simulate latency data from different pipeline configurations and analyze the distribution.

In [ ]:
import random

random.seed(42)

# Simulate latency data for 3 pipeline configurations
configs = {
    "baseline (no reranker)": [random.gauss(0.8, 0.15) for _ in range(50)],
    "with reranker": [random.gauss(1.5, 0.3) for _ in range(50)],
    "hybrid + reranker": [random.gauss(2.0, 0.4) for _ in range(50)],
}

for name, latencies in configs.items():
    metrics = compute_latency_metrics(latencies)
    print(f"\n{name}:")
    for key, val in metrics.items():
        print(f"  {key}: {val:.3f}s")

**Key takeaway:** Adding a reranker roughly doubles latency. Hybrid retrieval adds more on top. The question is whether the quality improvement justifies the latency cost.

## 2. Chunk Retrieval Metrics

How many chunks are retrieved per query, and how long are they? These metrics help diagnose retrieval quality.

In [ ]:
# Simulate chunk data for different chunking strategies
chunk_data = {
    "fixed_512": {
        "counts": [5, 5, 5, 4, 5, 3, 5, 5, 5, 5],
        "lengths": [[450, 512, 512, 480, 500], [512, 490, 512, 512, 500],
                    [512, 512, 400, 512, 512], [300, 512, 480, 512],
                    [512, 512, 512, 512, 450], [200, 512, 480],
                    [512, 512, 512, 512, 512], [512, 500, 480, 512, 512],
                    [512, 512, 512, 512, 512], [460, 512, 512, 512, 490]],
    },
    "semantic": {
        "counts": [3, 4, 2, 5, 3, 4, 3, 2, 4, 3],
        "lengths": [[800, 1200, 600], [500, 900, 700, 1100],
                    [1500, 800], [400, 600, 800, 1000, 500],
                    [900, 1100, 700], [600, 800, 1200, 400],
                    [1000, 800, 600], [1400, 900],
                    [500, 700, 900, 1100], [800, 1000, 600]],
    },
}

for strategy, data in chunk_data.items():
    metrics = compute_chunk_metrics(data["counts"], data["lengths"])
    print(f"\n{strategy}:")
    for key, val in metrics.items():
        print(f"  {key}: {val:.1f}")

**Observations:**
- Fixed chunking consistently returns the full top-k (5), while semantic chunking varies more
- Semantic chunks are larger on average because they group by topic boundaries
- More variation in semantic chunk counts means retrieval quality is more dependent on the query

## 3. Token Count and Cost Estimation

For cloud deployments (Bedrock, Vertex AI), cost scales with token count.

In [ ]:
from src.generate.prompt_templates import render_prompt

# Build prompts with different numbers of chunks
sample_chunks = [
    "The Great Wall of China stretches 21,196 km across northern China.",
    "Construction began in the 7th century BC during the Warring States period.",
    "The most well-known sections were built during the Ming Dynasty (1368-1644).",
    "It was designated a UNESCO World Heritage Site in 1987.",
    "The wall was built using stone, brick, tamped earth, and wood.",
]
question = "How long is the Great Wall?"

for template in ["default_qa", "structured", "chain_of_thought"]:
    for n_chunks in [1, 3, 5]:
        prompt = render_prompt(template, sample_chunks[:n_chunks], question)
        prompt_tokens = estimate_token_count(prompt)
        # Assume 100-token response
        cost = compute_cost_estimate(
            prompt_tokens, 100,
            prompt_price_per_1k=0.000035,  # Bedrock Nova Micro pricing
            completion_price_per_1k=0.00014,
        )
        print(f"{template:20s}  {n_chunks} chunks  {prompt_tokens:4d} tokens  ${cost['total_cost']:.6f}")

**Key insight:** On Bedrock Nova Micro pricing, even with 5 chunks and chain-of-thought prompting, each query costs fractions of a cent. Cost only matters at scale (thousands of queries/day) or with expensive models.

## 4. Understanding RAGAS Metrics

RAGAS evaluates RAG pipelines using LLM-as-judge. Here's what each metric measures:

| Metric | What It Measures | How It Works |
|---|---|---|
| **Faithfulness** | Is the answer grounded in the retrieved context? | LLM checks if each claim in the answer is supported by the context |
| **Answer Relevancy** | Does the answer address the question? | LLM generates questions from the answer; similarity to original question = score |
| **Context Precision** | Are the top-ranked chunks actually useful? | For each relevant chunk, how high is it ranked? |
| **Context Recall** | Did retrieval find all needed information? | Can the reference answer be reconstructed from retrieved chunks? |

These require a running LLM to compute. On Ollama locally, evaluation runs are slower but free.

## 5. Experiment Comparison Framework

When comparing experiments, look at metrics in this priority order:

In [ ]:
# Simulated experiment results for comparison
experiments = {
    "01_baseline": {
        "faithfulness": 0.72, "answer_relevancy": 0.78,
        "context_precision": 0.65, "context_recall": 0.70,
        "avg_latency": 0.82,
    },
    "02_semantic_chunking": {
        "faithfulness": 0.78, "answer_relevancy": 0.80,
        "context_precision": 0.72, "context_recall": 0.75,
        "avg_latency": 0.95,
    },
    "04_with_reranker": {
        "faithfulness": 0.85, "answer_relevancy": 0.83,
        "context_precision": 0.80, "context_recall": 0.72,
        "avg_latency": 1.50,
    },
    "05_hybrid_retrieval": {
        "faithfulness": 0.82, "answer_relevancy": 0.85,
        "context_precision": 0.78, "context_recall": 0.80,
        "avg_latency": 1.35,
    },
}

# Print comparison table
metrics = ["faithfulness", "answer_relevancy", "context_precision", "context_recall", "avg_latency"]
header = f"{'Experiment':<25}" + "".join(f"{m:<20}" for m in metrics)
print(header)
print("-" * len(header))
for name, scores in experiments.items():
    row = f"{name:<25}" + "".join(f"{scores[m]:<20.3f}" for m in metrics)
    print(row)

## 6. Identifying the Best Configuration

The "best" configuration depends on your priorities:

| Priority | Best Config | Why |
|---|---|---|
| **Lowest latency** | 01_baseline | No reranker overhead, fast responses |
| **Highest faithfulness** | 04_with_reranker | Reranker filters irrelevant chunks before LLM sees them |
| **Best recall** | 05_hybrid_retrieval | BM25 catches keyword matches that dense retrieval misses |
| **Best overall** | 05_hybrid_retrieval | Best balance of quality metrics with moderate latency |

## 7. Next Steps

Once you have real data from experiments:

1. **Run experiments** using `src/experiment_runner.py` to generate actual metrics
2. **Browse results** in MLflow UI (`mlflow ui` on port 5000)
3. **Set quality gates** — define minimum metric thresholds for deployment
4. **Track regressions** — compare each new experiment against the baseline
5. **Iterate** — change one variable at a time (chunker, embedder, reranker, prompt) to isolate what matters